# Transcript Matching — Model Diagnostics
Diagnostics for the production logistic-regression classifier trained by
`scripts/train_matching_model.py` (run that first).

**Inputs:** `data/model.pkl`, `data/best_thr.txt`, `data/pairs_of_transcripts/training_pairs.jsonl`  
**Outputs:** diagnostic plots


## 1. Imports & Setup

In [ ]:
import os
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve, confusion_matrix

module_path = os.path.abspath(os.path.join(''))
if module_path not in sys.path:
    sys.path.append(module_path)

if os.path.isdir("../data/"):
    os.chdir("../")

from notebooks.plots_utils import *
from scripts.train_matching_model import FEATURE_COLUMNS, MODEL_PATH, THRESHOLD_PATH, TRAINING_PAIRS_PATH

%load_ext autoreload
%autoreload 2

rcParams, column_width, page_width, shorts_style, RV_style = get_plt_attr()
plt.rcParams.update(rcParams)

## 2. Load Model Artefacts & Training Pairs

In [ ]:
with open(MODEL_PATH, 'rb') as f:
    model = pickle.load(f)
with open(THRESHOLD_PATH) as f:
    best_thr = float(f.read())

labeled_pairs = pd.read_json(TRAINING_PAIRS_PATH, lines=True, dtype={'videoId1': str, 'videoId2': str})
X_all = labeled_pairs[FEATURE_COLUMNS]
y_all = labeled_pairs['label_binary']

print(f"Total labeled pairs: {len(labeled_pairs)}, {int(y_all.sum())} positive and {int((y_all == 0).sum())} negative")
print(f"Best threshold: {best_thr:.4f}")
print("Model coefficients:")
for feat, coef in zip(model.feature_names_in_, model.coef_[0]):
    print(f"  {feat:30s} {coef:+.4f}")
print(f"  {'intercept':30s} {model.intercept_[0]:+.4f}")

In [ ]:
print("Class balance:")
display(labeled_pairs.groupby(['diffchannel', 'label_binary']).size().to_frame('count'))

## 3. Threshold Selection

In [ ]:
predicted_probas = model.predict_proba(X_all)[:, 1]
prec, rec, thr = precision_recall_curve(y_all, predicted_probas)
f1 = 2 * prec * rec / (prec + rec)

plt.figure(figsize=(6, 3))
plt.plot(thr, f1[1:], '.')
plt.axvline(best_thr, color='k', linestyle='--', label=f'best thr = {best_thr:.3f}')
plt.xlabel('Threshold')
plt.ylabel('F1')
plt.title('F1 vs threshold (train set)')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Diagnostics

In [ ]:
labeled_pairs['predicted_proba'] = predicted_probas
labeled_pairs['predicted_label'] = (predicted_probas >= best_thr).astype(int)

acc = (labeled_pairs['predicted_label'] == y_all).mean()
print(f"Train accuracy at best_thr: {acc:.4f}")
print("\nConfusion matrix (train):")
print(confusion_matrix(y_all, labeled_pairs['predicted_label']))

In [ ]:
# Decision-boundary histogram
fig, ax = plt.subplots(figsize=(column_width, 1.5))
bin_edges = np.linspace(0, 1, 50)

ax.axvspan(-0.01, best_thr, color='tab:blue', alpha=0.1)
ax.axvspan(best_thr, 1.01, color='tab:red',  alpha=0.1)

ax.hist(labeled_pairs[y_all == 0]['predicted_proba'], bins=bin_edges, label='non-matching pairs')
ax.hist(labeled_pairs[y_all == 1]['predicted_proba'], bins=bin_edges, label='matching pairs')
ax.axvline(best_thr, color='k', linestyle='--')

ax.set_xlabel('Predicted combined ratio')
ax.set_ylabel('Count')
ax.set_yscale('log')
ax.set_xlim(-0.01, 1.01)
ax.set_xticks(
    [0, 0.2, 0.4, 0.6, 0.8, 1, best_thr],
    ['0', '0.2', '0.4', '0.6', '0.8', '1', r'$\tau$' + f'\n(={best_thr:.2f})']
)
ax.text(best_thr / 2, 65, 'NEGATIVE', color='tab:blue', fontdict={'fontweight': 'bold'}, ha='center')
ax.text(best_thr + (1 - best_thr) / 2, 65, 'POSITIVE', color='tab:orange', fontdict={'fontweight': 'bold'}, ha='center')
ax.legend(loc=(0.2, 0.4))
plt.tight_layout(pad=0, w_pad=-0.5, h_pad=0)
fig.savefig('Figures/decision_boundary_plot.pdf', bbox_inches='tight')
plt.show()

In [ ]:
# Inspect misclassified pairs
misclassified_cols = [
    'label_binary', 'videoId1', 'videoId2',
    'transcript_clean1', 'transcript_clean2',
    'partialratio_cut_t1', 'partialratio_cut_t2',
    'sortratio_cut_t1', 'sortratio_cut_t2',
    'predicted_proba', 'predicted_label',
]
misclassified = labeled_pairs[
    labeled_pairs['predicted_label'] != labeled_pairs['label_binary']
][misclassified_cols]
print(f"{len(misclassified)} misclassified pairs")
display(misclassified)